# Service-Public XML Example

Notebook executable pour recuperer une fiche `service-public.fr` depuis le flux XML officiel DILA.

Exemple utilise: `F12391`.

Flux:
1. appel de l'API `data.gouv.fr` pour trouver le dataset,
2. recuperation du ZIP `vosdroits-latest.zip`,
3. extraction de `F12391.xml`,
4. conversion en markdown avec le parser XML du repo.


In [ ]:
from __future__ import annotations

import io
import json
import sys
import zipfile
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == 'scripts' else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from assistant_rh_data_engineering.service_public.xml_parser import parse_fiche_xml_from_bytes

DATA_GOUV_API_ROOT = 'https://www.data.gouv.fr/api/1'
DATASET_SLUG = 'service-public-fr-guide-vos-droits-et-demarches-particuliers'
DEFAULT_DIRECT_ZIP_URL = 'https://lecomarquage.service-public.gouv.fr/vdd/3.4/part/zip/vosdroits-latest.zip'
FICHE_ID = 'F12391'
OUTPUT_MARKDOWN = REPO_ROOT / 'tmp' / f'{FICHE_ID}.md'

print('Repo root:', REPO_ROOT)
print('Fiche ID :', FICHE_ID)
print('Markdown :', OUTPUT_MARKDOWN)

In [ ]:
def http_get(url: str) -> bytes:
    request = Request(
        url,
        headers={
            'User-Agent': 'assistant-rh/1.0 (+https://www.data.gouv.fr/)',
            'Accept': '*/*',
        },
    )
    with urlopen(request, timeout=60) as response:
        return response.read()


def fetch_dataset_metadata() -> dict:
    api_url = f'{DATA_GOUV_API_ROOT}/datasets/{DATASET_SLUG}/'
    return json.loads(http_get(api_url).decode('utf-8'))


def select_zip_url(dataset: dict) -> str:
    candidates = []
    for resource in dataset.get('resources', []):
        for key in ('url', 'latest', 'original_url'):
            value = resource.get(key)
            if isinstance(value, str) and value:
                candidates.append(value)

    for url in candidates:
        lowered = url.lower()
        if lowered.endswith('vosdroits-latest.zip') or '/zip/' in lowered:
            return url

    for url in candidates:
        if url.lower().endswith('.zip'):
            return url

    raise RuntimeError('Impossible de trouver une ressource ZIP dans les metadonnees data.gouv.fr.')


def load_fiche_xml_from_zip(zip_bytes: bytes, fiche_id: str) -> bytes:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        expected_suffixes = (f'/{fiche_id}.xml', f'\{fiche_id}.xml', f'{fiche_id}.xml')
        for member in archive.namelist():
            if member.endswith(expected_suffixes):
                return archive.read(member)
    raise FileNotFoundError(f'Fiche {fiche_id}.xml introuvable dans le ZIP.')

In [ ]:
try:
    dataset = fetch_dataset_metadata()
    zip_url = select_zip_url(dataset)
except (HTTPError, URLError, TimeoutError, RuntimeError) as exc:
    print(f"Fallback vers l'URL directe DILA: {exc}")
    zip_url = DEFAULT_DIRECT_ZIP_URL

zip_bytes = http_get(zip_url)
xml_bytes = load_fiche_xml_from_zip(zip_bytes, FICHE_ID)
parsed = parse_fiche_xml_from_bytes(xml_bytes, FICHE_ID)

assert parsed, f'Echec du parsing XML pour {FICHE_ID}'

print('ZIP utilise:', zip_url)
print('Titre:', parsed['title'])
print('URL source:', parsed['source_url'])
print('Date de verification:', parsed['metadata'].get('date_verification') or parsed.get('last_updated_date'))

In [ ]:
preview = parsed['doc_markdown'][:3000]
print(preview)

In [ ]:
OUTPUT_MARKDOWN.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_MARKDOWN.write_text(parsed['doc_markdown'], encoding='utf-8')
OUTPUT_MARKDOWN